# Automated promoter library redesign

這本 notebook 從 high-throughput PKL database 選取元件，不進行 de novo sequence generation。每個 element 的 model score 範圍切成五個等寬 bins；v1–v4 分別由 bin 1–4 選取，v5 是鎖定的 consensus，且 v1–v4 必須全部弱於 v5。

Spacer 是例外：`Spacer_v2` 來自 bin 2，`Spacer_v3 = Spacer_v2[:-2] + "TG"`，兩者是同一個 coupled design unit。

每個 design state 都組合成 `5^6 = 15,625` variants，使用固定且最大程度均勻的 64 種 3-bp gaps，再由 CorePromoter clean model 掃描最佳 register。

Validation：

- `shift != 0` 就計入 shifted variant。
- m10 shifted rate 與 m35 shifted rate 都必須 `< 10%`。
- 所有 shifted variants 都必須在 `-2..+2 bp`；任何 `abs(shift) > 2` 都不通過。
- 先最佳化 m10；m10 通過後作為 hard constraint，再最佳化 m35。
- 只有 global validation objective 嚴格改善才接受 replacement。


In [ ]:
# === Batch 0: setup and editable design config ===
import importlib
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

import automated_promoter_library_design as r
importlib.reload(r)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Select the frozen whole/CorePromoter checkpoint used for full-sequence scanning.
CORE_MODEL_VARIANT = "baseline"  # "baseline" or "tss_pas"
CORE_MODEL_CHECKPOINTS = {
    "baseline": r.WEIGHTS_DIR / "weights_CorePromoter_clean.pt",
    "tss_pas": r.WEIGHTS_DIR / "weights_CorePromoter_tss_pas.pt",
}
if CORE_MODEL_VARIANT not in CORE_MODEL_CHECKPOINTS:
    raise ValueError(f"Unknown CORE_MODEL_VARIANT: {CORE_MODEL_VARIANT!r}")
CORE_MODEL_CHECKPOINT = CORE_MODEL_CHECKPOINTS[CORE_MODEL_VARIANT]
if not CORE_MODEL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing {CORE_MODEL_VARIANT} checkpoint: {CORE_MODEL_CHECKPOINT}. "
        "Run Model_CorePromoter_TSS_pretrain.ipynb first."
    )

# RUN_MODE = "new": create a new output directory.
# RUN_MODE = "resume": continue an existing directory from its checkpoint/final_elements.
RUN_MODE = "new"  # "new" or "resume"
RESUME_DIR = r.DEFAULT_PARENT_OUT / "automated_redesign_20260716_112038"

if RUN_MODE == "new":
    OUT_DIR = r.DEFAULT_PARENT_OUT / f"automated_redesign_{RUN_STAMP}"
elif RUN_MODE == "resume":
    OUT_DIR = Path(RESUME_DIR)
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Resume directory does not exist: {OUT_DIR}")
else:
    raise ValueError(f"RUN_MODE must be 'new' or 'resume', not {RUN_MODE!r}")

CACHE_DIR = r.PROJECT_ROOT / "outputs" / "energy_bin_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Edit these sequences when the locked consensus changes.
CONSENSUS = {
    "UP": "TGGACTGATATATACAAAA",
    "m35": "TTGACA",
    "spacer": "TATGGGGCAAAATGGGG",
    "m10": "TATAAT",
    "DIS": "TTTTATTA",
    "ITS": "CAAAAAAAAG",
}

CONFIG = r.DesignConfig(
    consensus=CONSENSUS,
    bg5="CCCTTTCGTCTTCACACAGCAGCAGTCAGGTAGGGAAGAGACC",
    bg3="GTCGACTCTAGA",
    gap_length=3,
    gap_seed=777,
    random_seed=777,
    n_energy_bins=5,
    # Normalized observed-energy ranges used to create four mutable bins (v1-v4).
    # Tune the lower bound to trade weak-sequence coverage against register stability.
    mutable_energy_fraction_ranges={"m35": (0.3, 0.9), "m10": (0.3, 0.9)},
    max_candidates_per_unit=60,
    max_abs_shift=2,
    max_shift_rate=0.10,
    scan_batch_size=15625,
    require_derived_spacer_in_database=False,
)

USE_CACHE = True
MAX_SOURCE_ROWS = None  # None = use all observed sequences in each PKL

SEARCH_SETTINGS = {
    # This is an additional iteration allowance for each new/resume execution.
    "max_iterations": 150,
    "n_driver_units": 3,
    "probe_candidates_per_unit": 2,
    "pair_beam_width": 6,
    "max_pair_evaluations": 8,
    "max_stalled_iterations": 50,
}

config_filename = "run_config.json" if RUN_MODE == "new" else f"resume_config_{RUN_STAMP}.json"
r.save_run_config(
    OUT_DIR, CONFIG, SEARCH_SETTINGS, DEVICE,
    core_model_checkpoint=CORE_MODEL_CHECKPOINT,
    filename=config_filename,
)
print("Run mode:", RUN_MODE)
print("Project root:", r.PROJECT_ROOT)
print("Device:", DEVICE)
print("Output:", OUT_DIR)
print("Core model variant:", CORE_MODEL_VARIANT)
print("Core model checkpoint:", CORE_MODEL_CHECKPOINT)


## Batch 1: load trained models and build five equal-width energy bins

UP/Spacer/DIS/ITS 使用各自已訓練的 element weights。`Model_PL.ipynb` 的 -35/-10 data flow 使用 BPM，因此這兩個元素使用 `-BPM dG` 作為 higher-is-stronger score。不同 element 的分數不可互相比較。

Scored pools 會依 PKL 與 weight/BPM parameter 的檔案 signature 快取；來源或模型更新後會自動重建。


In [ ]:
element_models = r.ElementModelBundle(DEVICE)
core_model = r.load_core_model(DEVICE, checkpoint_path=CORE_MODEL_CHECKPOINT)

scored_pools = r.build_scored_pools(
    models=element_models,
    config=CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    max_source_rows=MAX_SOURCE_ROWS,
)

bin_summary = r.energy_bin_summary(scored_pools, CONFIG, element_models)
bin_summary.to_csv(OUT_DIR / "energy_bin_summary.csv", index=False)
display(bin_summary)


## Batch 2: construct database-backed design units

這個步驟執行 hard constraints：

- 一般 element 的 v1–v4 必須來自對應 bin 1–4。
- 所有 v1–v4 score 必須低於 locked v5。
- 同一 element 的五條 sequence 不可重複。
- Spacer_v2/v3 必須滿足 coupled rule。
- 任一必要 bin 沒有 eligible sequence 時直接報錯，不跨 bin 補候選。


In [ ]:
design_space = r.DesignSpace(
    config=CONFIG,
    models=element_models,
    scored_pools=scored_pools,
)

if RUN_MODE == "resume":
    saved_elements_path = OUT_DIR / "current_elements_checkpoint.csv"
    if not saved_elements_path.exists():
        saved_elements_path = OUT_DIR / "final_elements.csv"
    if not saved_elements_path.exists():
        raise FileNotFoundError(f"No resume elements found in {OUT_DIR}")
    initial_elements = pd.read_csv(saved_elements_path)
    initial_state = design_space.state_from_selected_elements(initial_elements)
    print("Recovered state from:", saved_elements_path)
else:
    initial_state = design_space.initial_state()
    initial_elements = design_space.selected_elements(initial_state)

unit_summary = design_space.candidate_pool_summary()

if RUN_MODE == "new":
    initial_elements.to_csv(OUT_DIR / "initial_elements.csv", index=False)
unit_summary.to_csv(OUT_DIR / "design_unit_candidate_counts.csv", index=False)
display(initial_elements)
display(unit_summary)
print("Candidate pool summary before search:")
print(unit_summary.to_string(index=False))


## Batch 3: fixed balanced 3-bp gap assignment and 15,625 assembly

`15,625 = 64 × 244 + 9`，所以無法完全等量。最大程度均勻的固定分配是 55 種 gap 各 244 次、9 種各 245 次。這份 assignment 在所有 replacement 前後保持不變。


In [ ]:
saved_gap_path = OUT_DIR / "gap_assignment.csv"
if RUN_MODE == "resume" and saved_gap_path.exists():
    gap_assignment = pd.read_csv(saved_gap_path)
    print("Loaded fixed gap assignment:", saved_gap_path)
else:
    gap_assignment = r.build_balanced_gap_assignment(CONFIG)

initial_variants = r.assemble_library(initial_elements, gap_assignment, CONFIG)

gap_counts = (
    gap_assignment["gap_3bp"]
    .value_counts()
    .rename_axis("gap_3bp")
    .reset_index(name="n_variants")
    .sort_values("gap_3bp")
)
gap_assignment.to_csv(OUT_DIR / "gap_assignment.csv", index=False)
if RUN_MODE == "new":
    initial_variants.to_csv(OUT_DIR / "initial_assembled_15625.csv", index=False)

print("Assembled variants:", f"{len(initial_variants):,}")
print("Gap count distribution:", gap_counts["n_variants"].value_counts().sort_index().to_dict())
display(gap_counts)


## Batch 4: initial CorePromoter register scan

正式欄位：

```text
m10_shift = observed_m10_start - design_m10_start
m35_shift = observed_m35_start - design_m35_start
spacer_length_shift = observed_spacer_len - design_spacer_len
```

Primary classification 使用 `m10_shift`；m10 通過後，secondary phase 使用 `m35_shift`。掃描同時輸出八個 architecture windows、observed element-role sequences、落在哪些 design regions，以及同一 element model 內的 delta energy。


In [ ]:
scanner = r.CorePromoterScanner(
    model=core_model,
    element_models=element_models,
    config=CONFIG,
    device=DEVICE,
)

initial_scan = scanner.scan(initial_variants, annotate_element_energies=True)
initial_summary = r.summarize_validation(initial_scan, CONFIG)
initial_scan.to_csv(OUT_DIR / "initial_scan_15625.csv", index=False)

validation_table = pd.DataFrame([
    {
        "anchor": anchor,
        "shifted_count": initial_summary[f"{anchor}_shifted_count"],
        "shifted_rate": initial_summary[f"{anchor}_shifted_rate"],
        "out_of_range_count": initial_summary[f"{anchor}_out_of_range_count"],
        "max_abs_shift": initial_summary[f"{anchor}_max_abs_shift"],
        "pass": initial_summary[f"{anchor}_validation_pass"],
    }
    for anchor in ("m10", "m35")
])
display(validation_table)
print("Global validation pass:", initial_summary["global_validation_pass"])


In [ ]:
# English-only plot labels for VSCode/Jupyter rendering stability.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=140)
for ax, anchor in zip(axes, ("m10", "m35")):
    counts = initial_scan[f"{anchor}_shift"].value_counts().sort_index()
    ax.bar(counts.index.astype(int), counts.values, color="#4C72B0")
    ax.axvspan(-2, 2, color="#55A868", alpha=0.15, label="Allowed magnitude")
    ax.set_title(f"{anchor} shift distribution")
    ax.set_xlabel("Shift (bp)")
    ax.set_ylabel("Variant count")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Batch 5: dominant shift and conditional-risk diagnosis

若目前 phase 仍有 `abs(shift) > 2`，優先診斷 out-of-range variants；否則診斷所有 `shift != 0` variants。先找數量最多的 shift coordinate，再以 `P(dominant shift | element version)` 排名 driver。

Observed role 落在某個 design region 只用來提供 redesign evidence；不同 element model 的 raw delta energy不互相比較。最終 replacement 仍由完整 15,625 before/after validation 決定。


In [ ]:
initial_diagnosis = r.diagnose_shift_drivers(
    scan=initial_scan,
    selected_elements=initial_elements,
    design_space=design_space,
    config=CONFIG,
)

print("Optimization phase:", initial_diagnosis["phase"])
print("Dominant shift coordinate:", initial_diagnosis["dominant_shift"])
display(initial_diagnosis["risk"].head(20))
display(initial_diagnosis["overlap"].head(30))


## Batch 6: monotonic automated redesign with live progress and resume

每輪會 print iteration、phase、目前 m10/m35 shifted rate、out-of-range count、dominant shift、driver units、每個 proposal 結果與接受/拒絕原因。

執行期間會在 `OUT_DIR` 持續覆寫：

- `search_progress.csv`：每輪狀態的持續更新 DataFrame。
- `proposal_history_checkpoint.csv`：所有已測 single/double proposals。
- `current_elements_checkpoint.csv`：目前接受的 30 條 sequences。
- `current_validation.json`：目前 validation。
- `search_checkpoint.json`：state、最後完成輪數、stalled counter 與已測 proposals。

`progress_df` 也會在記憶體中每輪原地更新；手動 interrupt 後可直接 `display(progress_df.tail())`。

若中途停止 kernel，將 setup cell 的 `RUN_MODE` 改為 `"resume"`，並把 `RESUME_DIR` 指向原本的 output directory；Batch 6 會從下一個 iteration 繼續。每次 resume 的 `max_iterations` 是額外輪數，不是總累積上限。


In [ ]:
redesigner = r.AutomatedRedesigner(
    design_space=design_space,
    scanner=scanner,
    gap_assignment=gap_assignment,
    config=CONFIG,
    out_dir=OUT_DIR,
    **SEARCH_SETTINGS,
)

# Mutated in place after every completed iteration. It remains available if
# this cell is manually interrupted.
progress_df = pd.DataFrame()

result = redesigner.run(
    initial_state=initial_state,
    initial_evaluation=(initial_elements, initial_scan, initial_summary),
    resume=(RUN_MODE == "resume"),
    reset_stalled_on_resume=True,
    progress_df=progress_df,
)

proposal_df = result.proposals

print("Success:", result.success)
print("Stop reason:", result.stop_reason)
print("Saved to:", result.out_dir)
display(pd.DataFrame([result.final_summary]))
display(progress_df)


## Batch 7: final design and audit tables

`final_elements.csv` 是最後保留的 30 條 element sequences；`proposal_history.csv` 同時保留 accepted 與 rejected moves，方便追蹤為何某次替換沒有被採用。


In [ ]:
display(result.final_elements)
display(result.final_risk.head(20))
display(result.final_overlap.head(30))

final_shift_table = pd.DataFrame([
    {
        "anchor": anchor,
        "shifted_count": result.final_summary[f"{anchor}_shifted_count"],
        "shifted_rate": result.final_summary[f"{anchor}_shifted_rate"],
        "out_of_range_count": result.final_summary[f"{anchor}_out_of_range_count"],
        "max_abs_shift": result.final_summary[f"{anchor}_max_abs_shift"],
        "pass": result.final_summary[f"{anchor}_validation_pass"],
    }
    for anchor in ("m10", "m35")
])
display(final_shift_table)

# English-only plot labels for VSCode/Jupyter rendering stability.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=140)
for ax, anchor in zip(axes, ("m10", "m35")):
    counts = result.final_scan[f"{anchor}_shift"].value_counts().sort_index()
    ax.bar(counts.index.astype(int), counts.values, color="#4C72B0")
    ax.axvspan(-2, 2, color="#55A868", alpha=0.15, label="Allowed magnitude")
    ax.set_title(f"Final {anchor} shift distribution")
    ax.set_xlabel("Shift (bp)")
    ax.set_ylabel("Variant count")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

accepted = result.proposals[result.proposals.get("accepted", False) == True] if len(result.proposals) else result.proposals
display(accepted)
